# AI4I 2020 — Failure Prediction Model Training

# AI4I 2020 — Model Training
Training XGBoost models for:
1. Binary failure prediction (with SMOTE + class weights)
2. Failure mode classifiers (TWF, HDF, PWF, OSF)
3. Threshold tuning for operational recall target
4. SHAP explainability validation
All trained models saved to ml/models/

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print("All imports successful")
print(f"XGBoost version: {__import__('xgboost').__version__}")
print(f"SHAP version: {shap.__version__}")

All imports successful
XGBoost version: 3.2.0
SHAP version: 0.51.0


This block is preparing the environment for the machine learning pipeline. 

It imports libraries for:
 data manipulation (Pandas, NumPy), 
 visualization (Matplotlib, Seaborn), 
 model training (XGBoost), 
 handling class imbalance (SMOTE), 
 evaluation (F1, ROC-AUC, Precision-Recall), 
 model explanation (SHAP), and
 file management (OS, Joblib, JSON). 
 
 It also configures display settings, creates output directories, suppresses unnecessary warnings, and verifies that critical packages are installed correctly before moving on to data loading and model training

Overall Purpose
Library Imports — Loads all packages required for data analysis, visualization, feature engineering, model training, evaluation, explainability, and file management.
Environment Configuration — Sets display formats, figure size, and plotting resolution.
Project Setup — Creates folders for saving models and processed outputs.
Verification — Confirms successful installation of critical libraries and displays their versions before starting the machine learning workflow

In [5]:

df = pd.read_csv('../data/raw/ai4i2020.csv')
print(f"Raw data loaded: {df.shape}")

# ── Feature Engineering ────────────────────────────────────
df['temp_diff']         = df['Process temperature [K]'] - df['Air temperature [K]']
df['power']             = df['Torque [Nm]'] * df['Rotational speed [rpm]'] * (2 * np.pi / 60)
df['tool_wear_torque']  = df['Tool wear [min]'] * df['Torque [Nm]']
df['quality_encoded']   = df['Type'].map({'H': 2, 'M': 1, 'L': 0})

#These are the predictor variables (inputs) used by the model.
FEATURE_COLS = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'temp_diff',
    'power',
    'tool_wear_torque',
    'quality_encoded'
]

FAILURE_MODE_COLS = ['TWF', 'HDF', 'PWF', 'OSF']  # skip RNF

TARGET_COL = 'Machine failure' #This is the label the model will predict. 0 or 1 indicating whether a machine failure occurred.


X = df[FEATURE_COLS].copy()#Creates a DataFrame containing only the selected features.
y = df[TARGET_COL].copy() #Stores the machine failure labels.
y_modes = df[FAILURE_MODE_COLS].copy()#Stores the individual failure types.

print(f"\nFeature matrix shape : {X.shape}")
print(f"Target shape         : {y.shape}")
print(f"Failure modes shape  : {y_modes.shape}")
print(f"\nFeature columns:")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"  {i}. {col}")
print(f"\nClass distribution: {dict(y.value_counts().sort_index())}") #Counts the number of samples in each target class.

Raw data loaded: (10000, 14)

Feature matrix shape : (10000, 9)
Target shape         : (10000,)
Failure modes shape  : (10000, 4)

Feature columns:
  1. Air temperature [K]
  2. Process temperature [K]
  3. Rotational speed [rpm]
  4. Torque [Nm]
  5. Tool wear [min]
  6. temp_diff
  7. power
  8. tool_wear_torque
  9. quality_encoded

Class distribution: {0: np.int64(9661), 1: np.int64(339)}


The code performs four main tasks:

Loads the AI4I predictive maintenance dataset.


Creates engineered features (temp_diff, power, tool_wear_torque, quality_encoded).


Builds training data:
X → model inputs
y → machine failure target
y_modes → specific failure types


Prints diagnostics such as shapes, feature names, and class distribution to verify the dataset is ready for machine learning.

In [8]:
# ── Stratified Split ───────────────────────────────────────
# stratify=y ensures failure rate is preserved in both splits
X_train, X_test, y_train, y_test, y_modes_train, y_modes_test = train_test_split(
    X, y, y_modes,
    test_size=0.2,
    random_state=42,
    stratify=y
) # x= feature matirx, y=machine failure labels, y_modes= individual failure types
# stratify=y ensures that the proportion of machine failures (the target variable) is preserved in both the training and test sets. This is crucial for imbalanced datasets to ensure that the model learns from a representative sample of both classes.
print("TRAIN/TEST SPLIT (stratified)")

print(f"Training set   : {X_train.shape[0]:,} rows")
print(f"Test set       : {X_test.shape[0]:,} rows")
print(f"\nFailure rate in training : {y_train.mean()*100:.2f}%")
print(f"Failure rate in test     : {y_test.mean()*100:.2f}%")
print(f"\nFailures in training set : {y_train.sum()}")
print(f"Failures in test set     : {y_test.sum()}")
print("\nFailure rate preserved in both splits — stratification worked")

TRAIN/TEST SPLIT (stratified)
Training set   : 8,000 rows
Test set       : 2,000 rows

Failure rate in training : 3.39%
Failure rate in test     : 3.40%

Failures in training set : 271
Failures in test set     : 68

Failure rate preserved in both splits — stratification worked


In [11]:
#feature scaling (standardization) 

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # transform only, don't fit becuase the test set must remain completely unseen.


#fit()      -> learn parameters
#transform() -> apply those parameters


# Convert back to DataFrame to keep column names for SHAP
#why?=> fit_transform returns array without column names, but we need them later for SHAP explanations. Converting back to DataFrame allows us to retain the feature names.

X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=FEATURE_COLS)

print("FEATURE SCALING")
print("=" * 50)
print("Scaler fit on training data only")
print(f"\nPre-scaling  — Torque mean : {X_train['Torque [Nm]'].mean():.2f}")
print(f"Post-scaling — Torque mean : {X_train_scaled['Torque [Nm]'].mean():.4f}  (should be ~0)")
print(f"Post-scaling — Torque std  : {X_train_scaled['Torque [Nm]'].std():.4f}   (should be ~1)")

# Save scaler immediately — needed for production inference
joblib.dump(scaler, '../models/scaler_ai4i.joblib')
print("\n✅ Scaler saved → ml/models/scaler_ai4i.joblib")

FEATURE SCALING
Scaler fit on training data only

Pre-scaling  — Torque mean : 40.00
Post-scaling — Torque mean : -0.0000  (should be ~0)
Post-scaling — Torque std  : 1.0001   (should be ~1)

✅ Scaler saved → ml/models/scaler_ai4i.joblib


In [12]:
# ── SMOTE Oversampling ─────────────────────────────────────
# Applied ONLY on training data
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE OVERSAMPLING")
print("=" * 50)
print(f"Before SMOTE:")
print(f"  Training rows     : {len(X_train_scaled):,}")
print(f"  Failures (1)      : {y_train.sum():,}  ({y_train.mean()*100:.2f}%)")
print(f"  No failures (0)   : {(y_train==0).sum():,}")

print(f"\nAfter SMOTE:")
print(f"  Training rows     : {len(X_train_resampled):,}")
unique, counts = np.unique(y_train_resampled, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}           : {cnt:,}  ({cnt/len(y_train_resampled)*100:.2f}%)")

print("\n✅ Classes now balanced for training")
print("⚠️  Test set remains UNTOUCHED — real-world distribution preserved")

SMOTE OVERSAMPLING
Before SMOTE:
  Training rows     : 8,000
  Failures (1)      : 271  (3.39%)
  No failures (0)   : 7,729

After SMOTE:
  Training rows     : 15,458
  Class 0           : 7,729  (50.00%)
  Class 1           : 7,729  (50.00%)

✅ Classes now balanced for training
⚠️  Test set remains UNTOUCHED — real-world distribution preserved
